# Uncertainty-TTA on Kaggle — FULL SWEEP (all backbones × all datasets)

A reproduce-everything driver for Kaggle, built to make **cross-hardware comparisons**
possible: run the whole pipeline here on a Kaggle GPU (e.g. T4) and compare the
resulting `inference_benchmark.csv` against the local run (e.g. an RTX 4080). Same
code path as local — everything goes through `scripts/run_all.py`.

This is the *general* notebook. For a single-dataset workflow scoped to one student's
assignment, use `kaggle_phase5_effb0.ipynb` instead (DermaMNIST / S2).

**Kaggle data-handling cell contributed by S2 (Mohamed Abdel Sattar)** — reused as-is.

> ⚠️ Re-upload the **current** repo as your Kaggle Dataset first — these cells need the
> Phase 5 code (`--arch`, Top-K, `run_all.py`). An old Phase-3 snapshot will fail.

> ⏱️ **Kaggle sessions are time-limited (~9–12 h).** The full 2-backbone × 6-dataset ×
> 3-seed sweep — PathMNIST especially — may not finish in one session. Everything below
> uses `--skip-existing`, so a run that's cut off **resumes** where it left off on the
> next session (as long as your `checkpoints/` persist — see the export/restore note at
> the bottom). Start with the small datasets; give PathMNIST its own session.

## 1. Verify GPU (note the name — it labels your benchmark)

In [ ]:
import torch
print('PyTorch    :', torch.__version__)
print('CUDA avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU        :', torch.cuda.get_device_name(0))
else:
    print('⚠ No GPU detected. Enable it in Settings → Accelerator before continuing.')

## 2. Attach the repo + install dependencies

`torch`/`torchvision` ship with the Kaggle GPU image; we add `medmnist`. The next cell
is S2's Kaggle data-handler: it finds the repo Dataset under `/kaggle/input`, copies it
to the writable working dir, `chdir`s in, and clears cached `src.*` imports.

In [ ]:
!pip install -q medmnist==3.0.2 2>/dev/null

In [ ]:
from pathlib import Path
import shutil
import os
import sys

# =========================================================
# Change these only
# =========================================================
PROJECT_FOLDER_NAME = "uncertainty-tta-medmnist-main"

# =========================================================
# Kaggle roots
# =========================================================
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")

# =========================================================
# Find project folder inside /kaggle/input
# =========================================================
project_matches = [p for p in INPUT_ROOT.rglob(PROJECT_FOLDER_NAME) if p.is_dir()]

if not project_matches:
    raise FileNotFoundError(f"Could not find project folder: {PROJECT_FOLDER_NAME}")

project_src = project_matches[0]
project_dst = WORKING_ROOT / PROJECT_FOLDER_NAME

print("Project source:", project_src)
print("Project destination:", project_dst)

# =========================================================
# Copy project to /kaggle/working
# =========================================================
if project_dst.exists():
    shutil.rmtree(project_dst)

shutil.copytree(project_src, project_dst)

# =========================================================
# Move into project folder
# =========================================================
os.chdir(project_dst)

# =========================================================
# Remove old cached src imports
# =========================================================
for m in list(sys.modules):
    if m == "src" or m.startswith("src."):
        del sys.modules[m]

# =========================================================
# Add project root to Python path
# =========================================================
PROJECT_DIR = str(project_dst)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# =========================================================
# Confirm setup
# =========================================================
print("\nCurrent directory:")
print(os.getcwd())

print("\nPython path:")
print(sys.path[:3])

print("\nProject files:")
print(os.listdir("."))

## 3. Configure the sweep

Edit these, then run the cell below. Defaults = the full sweep. For an **inference-only
comparison** (fastest path to a benchmark you can diff against local), use the commented
"light" config: it trains one seed per (backbone, dataset) and benchmarks — no 3-seed
study, no ablations.

In [ ]:
# ---- FULL SWEEP (default) ----
MODELS   = ['resnet18', 'effb0']
DATASETS = ['breastmnist', 'pneumoniamnist', 'dermamnist', 'bloodmnist', 'organamnist', 'pathmnist']
SEEDS    = [0, 42, 123]
PHASES   = ['train', 'weighted', 'standard', 'ablate', 'strips', 'benchmark', 'aggregate']

# ---- LIGHT: inference comparison only (uncomment to use) ----
# MODELS   = ['resnet18', 'effb0']
# DATASETS = ['breastmnist', 'pneumoniamnist', 'dermamnist', 'bloodmnist', 'organamnist', 'pathmnist']
# SEEDS    = [42]
# PHASES   = ['train', 'benchmark']

NWORK = 2   # Kaggle is Linux; workers>0 is fine and faster than 0

print('models  :', MODELS)
print('datasets:', DATASETS)
print('seeds   :', SEEDS)
print('phases  :', PHASES)

## 4. Preview the plan (no compute)

`--dry-run` prints every step run_all will execute and the total count, without touching
the GPU. Sanity-check the scope and rough size before committing a session to it.

In [ ]:
mstr = ' '.join(MODELS); dstr = ' '.join(DATASETS); sstr = ' '.join(map(str, SEEDS)); pstr = ' '.join(PHASES)
!python -m scripts.run_all --models {mstr} --datasets {dstr} --seeds {sstr} --phases {pstr} \
    --num-workers {NWORK} --skip-existing --continue-on-error --dry-run

## 5. Run the sweep

`--skip-existing` skips any training whose checkpoint already exists (so this is
resumable), and `--continue-on-error` pushes through a failed step. run_all writes a
per-step wall-clock to `results/run_manifest.csv`.

Tip: if a session is about to time out, just re-run this cell next session — finished
checkpoints are skipped and it continues. To go even bigger you can append `aggregate`,
or the global cross-dataset phases (`matrix significance reliability analysis`) once all
six datasets exist.

In [ ]:
!python -m scripts.run_all --models {mstr} --datasets {dstr} --seeds {sstr} --phases {pstr} \
    --num-workers {NWORK} --skip-existing --continue-on-error

## 6. Inference benchmark — labeled for cross-hardware comparison

`run_all` already produced `results/inference_benchmark.csv` (per backbone × N × method).
Here we write a **device-tagged copy** so when you merge it with the local CSV you can tell
the two GPUs apart. `tta_all_strategies` is the shared row for every soft strategy and
every Top-K at a given N.

In [ ]:
import pandas as pd, torch, re
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
gpu_slug = re.sub(r'[^A-Za-z0-9]+', '_', gpu).strip('_').lower()

bench = pd.read_csv('results/inference_benchmark.csv')
bench.insert(0, 'device', gpu)                      # label the hardware
out = f'results/inference_benchmark_{gpu_slug}.csv'
bench.to_csv(out, index=False)
print(f'Wrote {out}  (device = {gpu})')

# Quick comparable view: ms/image by backbone × method × N on THIS hardware.
piv = bench.pivot_table(index=['arch', 'method'], columns='n_views',
                        values='ms_per_image', aggfunc='mean').round(3)
display(piv)
print('\nTo compare with local: load both inference_benchmark_<device>.csv files and '
      'diff on [arch, method, n_views]. The device column keeps the two machines distinct.')

## 7. Sweep summary + runtime manifest

What finished, and how long each step took on this hardware.

In [ ]:
import pandas as pd, os
man = pd.read_csv('results/run_manifest.csv')
print('Steps:', len(man), '| ok:', (man.status == 'ok').sum(),
      '| skipped:', (man.status == 'skipped').sum(),
      '| failed:', man.status.str.startswith('FAIL').sum())
print('\nRuntime by phase (s):')
display(man.groupby('phase')['seconds'].sum().sort_values(ascending=False).round(1))

# effb0 mean±std if aggregate ran
p = 'results/effb0_seed_stability.csv'
if os.path.exists(p):
    print('\nEfficientNet-B0 mean ± std:')
    display(pd.read_csv(p).round(4))

## 8. Download artifacts (+ how to resume next session)

Zip checkpoints, results, figures, predictions. **To resume a cut-off sweep:** download
this zip, and on your next Kaggle session either (a) re-attach it / unzip its
`checkpoints/` into the working dir before running, or (b) commit checkpoints to your
Drive — `--skip-existing` then skips everything already trained. The device-tagged
benchmark CSV is what you diff against the local run.

In [ ]:
!zip -r -q /kaggle/working/tta_full_sweep.zip checkpoints results figures predictions
print('Wrote /kaggle/working/tta_full_sweep.zip — download from the Output panel.')